# 大涨大跌逐日一致性截图诊断

本 Notebook 只读取 `runtime_outputs/` 中已经生成的结果，不会重新运行模型、不重建候选、不读取或修改三状态结果。

请在远端验证包内运行本 Notebook。运行完成后，给出下面代码输出的截图即可：

1. 总体状态表；
2. 大跌侧和大涨侧的字段级不一致统计；
3. 如果存在差异，给出前几条差异示例。

行级 `MISMATCH` 只表示这一行至少有一个字段不同，不等于预测一定错了。字段级表会区分 `close`、`score`、`predicted`、实际极端标签、O2O 和阶段等字段。

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

def find_package_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    candidates.extend([
        Path('/home/hzy/cta/05_上传包_日期解析修正版'),
        Path('/hpfs/innofs/home/hzy/cta/05_上传包_日期解析修正版'),
    ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / 'src' / 'remote_validation.py').is_file() and (candidate / 'runtime_outputs').is_dir():
            return candidate
    raise FileNotFoundError(
        '没有找到本上传包根目录。请确认 Notebook 放在 05_上传包_日期解析修正版 内，且 runtime_outputs 已由主入口生成。'
    )

PACKAGE_ROOT = find_package_root(Path.cwd())
OUTPUT_DIR = Path(os.environ.get('UPLOAD_OUTPUT_DIR', str(PACKAGE_ROOT / 'runtime_outputs'))).expanduser().resolve()
MANIFEST_PATH = OUTPUT_DIR / '最终一致性结论.json'

print('上传包根目录：', PACKAGE_ROOT)
print('结果目录：    ', OUTPUT_DIR)
print('结果目录存在：', OUTPUT_DIR.is_dir())
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'没有找到 {MANIFEST_PATH}。请先运行 00_主入口_远端验证.ipynb。')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
summary = manifest.get('execution_summary', {})
print('manifest success：', manifest.get('success'))
print('简表 valid：      ', summary.get('valid'))
print('简表日期范围：    ', summary.get('date_min'), '→', summary.get('date_max'))
print('简表行数：        ', summary.get('rows'))

In [ ]:
FIELD_ORDER = [
    'close', 'score', 'predicted', 'actual_extreme',
    'correct', 'direction_correct', 'o2o_bp',
    'signed_o2o_bp', 'phase',
]
FIELD_EXPLANATION = {
    'close': '收盘价',
    'score': '模型分数',
    'predicted': '预测标签',
    'actual_extreme': '实际极端标签',
    'correct': '预测是否命中',
    'direction_correct': '方向是否正确',
    'o2o_bp': '实际 O2O(bp)',
    'signed_o2o_bp': '方向化 O2O(bp)',
    'phase': '阶段',
}

def as_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype('string').str.strip().str.lower().isin(['true', '1', 'yes'])

def load_compare(side: str) -> pd.DataFrame:
    path = OUTPUT_DIR / f'大涨大跌_{side}_逐日对比.csv'
    if not path.is_file():
        raise FileNotFoundError(f'没有找到逐日对比文件：{path}。请先运行主入口。')
    return pd.read_csv(path, encoding='utf-8-sig')

overall_rows = []
for side, label in [('down', '大跌'), ('up', '大涨')]:
    frame = load_compare(side)
    common = frame['_merge'].astype('string').eq('both')
    all_match = as_bool(frame['all_columns_match'])
    overall_rows.append({
        '方向': label,
        '远端行数': len(frame),
        '共同日期数': int(common.sum()),
        '日期仅远端': int(frame['_merge'].astype('string').eq('left_only').sum()),
        '日期仅本地': int(frame['_merge'].astype('string').eq('right_only').sum()),
        '整行不一致数': int((common & ~all_match).sum()),
        '整行完全一致': bool(common.all() and all_match[common].all()),
    })

print('===== 总体状态：请优先截图这一张 =====')
display(pd.DataFrame(overall_rows))

In [ ]:
for side, label in [('down', '大跌'), ('up', '大涨')]:
    frame = load_compare(side)
    common = frame['_merge'].astype('string').eq('both')
    field_rows = []
    for field in FIELD_ORDER:
        match_col = f'match_{field}'
        if match_col not in frame.columns:
            continue
        matched = as_bool(frame[match_col])
        field_rows.append({
            '字段': field,
            '含义': FIELD_EXPLANATION[field],
            '不一致行数': int((common & ~matched).sum()),
            '一致行数': int((common & matched).sum()),
        })

    print(f'===== {label}：字段级不一致统计 =====')
    field_table = pd.DataFrame(field_rows).sort_values(['不一致行数', '字段'], ascending=[False, True])
    display(field_table)

    all_match = as_bool(frame['all_columns_match'])
    bad = frame.loc[common & ~all_match].copy()
    if bad.empty:
        print(f'{label}没有共同日期上的字段差异。')
        continue

    examples = []
    for _, row in bad.head(8).iterrows():
        for field in FIELD_ORDER:
            match_col = f'match_{field}'
            if match_col not in frame.columns or bool(as_bool(pd.Series([row[match_col]])).iloc[0]):
                continue
            examples.append({
                '日期': row.get('date'),
                '字段': field,
                '远端值': row.get(f'{field}_generated'),
                '本地值': row.get(f'{field}_local'),
            })
    print(f'===== {label}：前 8 条不一致日期中的差异字段 =====')
    display(pd.DataFrame(examples))

print('截图提示：如果只能发截图，请至少发“总体状态”和两张“字段级不一致统计”表。')